# Experiment 1 - No-Tune


## Install & Import Library

In [ ]:
# install libraryt
!pip install statsmodels yfinance pandas-datareader requests scikit-learn xgboost lightgbm optuna plotly --quiet

In [ ]:
# import some libarries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
import os
import json
import pickle
import random

import requests
import re
import yfinance as yf
from datetime import datetime

from openpyxl import load_workbook

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

import xgboost as xgb
import lightgbm as lgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

In [ ]:
# set reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed = 42
set_seed(seed)

## Konfigurasi

In [ ]:
# path dataset
PATH_ANTAM = '/kaggle/input/datasets/naufalfz/dataset/harga_emas_antm.csv'
PATH_XAU = '/kaggle/input/datasets/naufalfz/dataset/Data Historis XAU_USD.csv'
PATH_KURS = '/kaggle/input/datasets/naufalfz/dataset/data kurs usd idr.xlsx'
PATH_BI = '/kaggle/input/datasets/naufalfz/dataset/BI-7Day-RR.xlsx'
PATH_INFLASI = '/kaggle/input/datasets/naufalfz/dataset/Data Inflasi.xlsx'

In [ ]:
# config crawling, feature engineering params modeling
START_DATE = '2019-01-01'
END_DATE = '2026-02-20'
TRAIN_RATIO = 0.70
MAX_HORIZON = 7
SARIMAX_ORDER = (1, 1, 1)
SARIMAX_SEAS = (0, 0, 0, 0)
N_OPTUNA_TRIALS = 50

## Load & Merge Dataset

In [ ]:
BULAN_ID = {
    'Januari':'01','Februari':'02','Maret':'03','April':'04',
    'Mei':'05','Juni':'06','Juli':'07','Agustus':'08',
    'September':'09','Oktober':'10','November':'11','Desember':'12'
}

def parse_tgl_id(s):
    p = s.strip().split()
    return pd.to_datetime(f"{p[2]}-{BULAN_ID[p[1]]}-{p[0].zfill(2)}")

def parse_periode_id(s):
    p = s.strip().split()
    return pd.to_datetime(f"{p[1]}-{BULAN_ID[p[0]]}-01")

# 1. Harga Emas Antam
df_antam = pd.read_csv(PATH_ANTAM)
df_antam['tanggal'] = pd.to_datetime(df_antam['tanggal'])
df_antam = df_antam[['tanggal','harga']].rename(columns={'harga':'harga_emas_antam_idr'}).set_index('tanggal').sort_index()

# 2. XAU/USD (Investing.com)
df_xau = pd.read_csv(PATH_XAU)
df_xau['Tanggal'] = pd.to_datetime(df_xau['Tanggal'], format='%d/%m/%Y')
df_xau['harga_emas_dunia_usd'] = (df_xau['Terakhir']
    .str.replace('.','',regex=False).str.replace(',','.',regex=False).astype(float))
df_xau = df_xau[['Tanggal','harga_emas_dunia_usd']].set_index('Tanggal').sort_index()

# 3. Kurs USD/IDR (yfinance)
df_kurs = pd.read_excel(PATH_KURS)
df_kurs.drop('Unnamed: 0', axis=1, inplace=True)
df_kurs.columns = ['tanggal','kurs_usdidr']
df_kurs['tanggal'] = pd.to_datetime(df_kurs['tanggal'])
df_kurs = df_kurs[['tanggal','kurs_usdidr']].set_index('tanggal').sort_index()

# 4. BI 7-Day RR Rate (Bank Indonesia)
wb_bi = load_workbook(PATH_BI, read_only=True); ws_bi = wb_bi.active
bi_rows = []
for row in ws_bi.iter_rows(values_only=True):
    if row[0] is not None and isinstance(row[0], int):
        bi_rows.append({'tanggal': parse_tgl_id(row[1]),
                        'bi_7drr_rate': float(row[2].replace('%','').strip())})
df_bi = pd.DataFrame(bi_rows).set_index('tanggal').sort_index()

# 5. Inflasi YoY (Bank Indonesia)
wb_inf = load_workbook(PATH_INFLASI, read_only=True); ws_inf = wb_inf.active
inf_rows = []
for row in ws_inf.iter_rows(values_only=True):
    if row[0] is not None and isinstance(row[0], int):
        inf_rows.append({'tanggal': parse_periode_id(row[1]),
                         'inflasi_yoy_id': float(row[2].replace('%','').strip())})
df_inflasi = pd.DataFrame(inf_rows).set_index('tanggal').sort_index()

In [ ]:
# merge ke daily index
full_idx = pd.date_range(start=START_DATE, end=END_DATE, freq='D')

df = pd.DataFrame(index=full_idx)
df.index.name = 'tanggal' # set index

# join seluruh data
df = df.join(df_antam).join(df_xau).join(df_kurs)
df = df.join(df_bi.reindex(full_idx).ffill().bfill())
df = df.join(df_inflasi.reindex(full_idx).ffill().bfill())
df = df.ffill().bfill() # fillna metode (ffill dan bfill)

# filter hari kerja (Senin–Jumat) (oposional)
# df = df[df.index.dayofweek < 5].copy()

## Uji Stasioneritas (ADF Test)

In [ ]:
# function adf test (cek stasioner)
def adf_test(series, name):
    res = adfuller(series.dropna(), autolag='AIC')
    status = 'Stasioner' if res[1] < 0.05 else 'Tidak Stasioner'
    print(f'{name:<35} | ADF: {res[0]:>8.4f} | p: {res[1]:.4f} | {status}')

print('Level:')
adf_test(df['harga_emas_antam_idr'], 'Harga Emas Antam (level)')
adf_test(df['harga_emas_dunia_usd'], 'Harga Emas Dunia (level)')
adf_test(df['kurs_usdidr'], 'Kurs USD/IDR (level)')

df['log_harga'] = np.log(df['harga_emas_antam_idr'])
df['log_xau'] = np.log(df['harga_emas_dunia_usd'])
df['log_kurs'] = np.log(df['kurs_usdidr'])

print('\nLog')
adf_test(df['log_harga'], 'log(Harga Emas Antam)')
adf_test(df['log_xau'], 'log(Harga Emas Dunia)')
adf_test(df['log_kurs'], 'log(Kurs USD/IDR)')

print('\nLog First Difference')
adf_test(df['log_harga'].diff().dropna(), 'log(Harga Emas Antam)')
adf_test(df['log_xau'].diff().dropna(), 'log(Harga Emas Dunia)')
adf_test(df['log_kurs'].diff().dropna(), 'log(Kurs USD/IDR)')

Level:
Harga Emas Antam (level)            | ADF:   5.0517 | p: 1.0000 | Tidak Stasioner
Harga Emas Dunia (level)            | ADF:   5.6124 | p: 1.0000 | Tidak Stasioner
Kurs USD/IDR (level)                | ADF:  -1.6036 | p: 0.4818 | Tidak Stasioner

Log
log(Harga Emas Antam)               | ADF:   2.5870 | p: 0.9991 | Tidak Stasioner
log(Harga Emas Dunia)               | ADF:   2.3568 | p: 0.9990 | Tidak Stasioner
log(Kurs USD/IDR)                   | ADF:  -1.6935 | p: 0.4345 | Tidak Stasioner

Log First Difference
log(Harga Emas Antam)               | ADF: -14.7073 | p: 0.0000 | Stasioner
log(Harga Emas Dunia)               | ADF: -17.2375 | p: 0.0000 | Stasioner
log(Kurs USD/IDR)                   | ADF:  -8.9902 | p: 0.0000 | Stasioner


## Feature Engineering

In [ ]:
# Log return
df['log_return'] = df['log_harga'].diff()

# Lag features
for lag in [1, 2, 3, 5, 7]:
    df[f'lag_{lag}'] = df['log_harga'].shift(lag)

# Rolling statistics
df['rolling_mean_7']  = df['log_harga'].shift(1).rolling(7).mean()
df['rolling_std_7']   = df['log_harga'].shift(1).rolling(7).std()
df['rolling_mean_14'] = df['log_harga'].shift(1).rolling(14).mean()
df['rolling_std_14']  = df['log_harga'].shift(1).rolling(14).std()

# Calendar features
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['quarter'] = df.index.quarter

# Macro spread
df['spread_macro'] = df['bi_7drr_rate'] - df['inflasi_yoy_id']

# Lag eksogen
df['log_xau_lag1'] = df['log_xau'].shift(1)
df['log_kurs_lag1'] = df['log_kurs'].shift(1)

# Drop NaN akibat diff, lag, rolling
df = df.dropna().copy()

In [ ]:
# fitur yg diambil
FEATURE_COLS = [
    'log_return',
    'lag_1','lag_2','lag_3','lag_5','lag_7',
    'rolling_mean_7','rolling_std_7','rolling_mean_14','rolling_std_14',
    'log_xau','log_kurs','log_xau_lag1','log_kurs_lag1',
    'bi_7drr_rate','inflasi_yoy_id','spread_macro',
    'day_of_week','month','quarter'
]

EXOG_SARIMAX = ['log_xau', 'log_kurs']  # eksogen SARIMAX: XAU/USD & Kurs saja

print(f'Shape setelah feature engineering: {df.shape}')
print(f'SARIMAX exogenous : {EXOG_SARIMAX}')
print(f'Boosting features : {len(FEATURE_COLS)} kolom')
df[FEATURE_COLS].head()

Shape setelah feature engineering: (2580, 24)
SARIMAX exogenous : ['log_xau', 'log_kurs']
Boosting features : 20 kolom


,log_return,lag_1,lag_2,lag_3,lag_5,lag_7,rolling_mean_7,rolling_std_7,rolling_mean_14,rolling_std_14,log_xau,log_kurs,log_xau_lag1,log_kurs_lag1,bi_7drr_rate,inflasi_yoy_id,spread_macro,day_of_week,month,quarter
tanggal,,,,,,,,,,,,,,,,,,,,
2019-01-29,-0.004515,13.409045,13.412043,13.412043,13.399995,13.399995,13.404729,0.006241,13.403874,0.004530,7.179255,9.551303,7.172831,9.552369,6.0,2.82,3.18,1,1,1
2019-01-30,0.006015,13.404530,13.409045,13.412043,13.396960,13.403021,13.405377,0.005894,13.404198,0.004391,7.185319,9.552084,7.179255,9.551303,6.0,2.82,3.18,2,1,1
2019-01-31,0.005979,13.410545,13.404530,13.409045,13.412043,13.399995,13.406452,0.006076,13.404843,0.004624,7.186099,9.549167,7.185319,9.552084,6.0,2.82,3.18,3,1,1
2019-02-01,-0.001491,13.416524,13.410545,13.404530,13.412043,13.396960,13.408813,0.006354,13.405700,0.005575,7.183871,9.544381,7.186099,9.549167,6.0,2.57,3.43,4,2,1
2019-02-02,0.000000,13.415033,13.416524,13.410545,13.409045,13.412043,13.411395,0.003953,13.406450,0.006088,7.183871,9.544381,7.183871,9.544381,6.0,2.57,3.43,5,2,1


In [ ]:
# splitting data
split_idx = int(len(df) * TRAIN_RATIO)
train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

print(f'Train : {len(train):>5} baris | {train.index[0].date()} – {train.index[-1].date()}')
print(f'Test : {len(test):>5} baris | {test.index[0].date()} – {test.index[-1].date()}')

Train :  1805 baris | 2019-01-29 – 2024-01-07
Test :   775 baris | 2024-01-08 – 2026-02-20


In [ ]:
# helper function
def evaluate_forecast(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred)
    }

tscv = TimeSeriesSplit(n_splits=5)


In [ ]:
# default params tanpa tuning
default_xgb_params = {'random_state': seed, 'n_jobs': -1}
default_lgb_params = {'random_state': seed, 'n_jobs': -1, 'verbose': -1}

all_results_notune = []
metrics_rows_notune = []

for h in range(1, MAX_HORIZON + 1):
    print(f'(Tanpa Tuning) Horizon H+{h}')

    train_pairs = []
    for t in range(len(train) - h):
        history_y = train['log_harga'].iloc[:t+1]
        history_exog = train[EXOG_SARIMAX].iloc[:t+1]
        future_exog  = train[EXOG_SARIMAX].iloc[t+1:t+1+h]
        if len(future_exog) < h:
            continue
        try:
            m_hist = SARIMAX(
                history_y, exog=history_exog,
                order=SARIMAX_ORDER, seasonal_order=SARIMAX_SEAS,
                enforce_stationarity=False, enforce_invertibility=False, freq='D'
            ).fit(disp=False, maxiter=200)
            fc_h = m_hist.forecast(steps=h, exog=future_exog).iloc[-1]
            residual_h = train['log_harga'].iloc[t + h] - fc_h
            train_pairs.append((train[FEATURE_COLS].iloc[t].values, residual_h))
        except Exception:
            continue

    X_tr = np.array([p[0] for p in train_pairs])
    y_tr = np.array([p[1] for p in train_pairs])

    xgb_h_nt = xgb.XGBRegressor(**default_xgb_params)
    xgb_h_nt.fit(X_tr, y_tr)
    lgb_h_nt = lgb.LGBMRegressor(**default_lgb_params)
    lgb_h_nt.fit(X_tr, y_tr)

    history_wf = train.copy()
    sarimax_preds_log, actual_vals, feat_rows = [], [], []

    for i in range(len(test) - h + 1):
        exog_future = test[EXOG_SARIMAX].iloc[i:i+h]
        if len(exog_future) < h:
            break
        try:
            m = SARIMAX(
                history_wf['log_harga'], exog=history_wf[EXOG_SARIMAX],
                order=SARIMAX_ORDER, seasonal_order=SARIMAX_SEAS,
                enforce_stationarity=False, enforce_invertibility=False, freq='D'
            ).fit(disp=False, maxiter=200)
            fc = m.forecast(steps=h, exog=exog_future)
            sarimax_preds_log.append(fc.iloc[-1])
            feat_rows.append(test[FEATURE_COLS].iloc[i].values)
            actual_vals.append(test['harga_emas_antam_idr'].iloc[i + h - 1])
            history_wf = pd.concat([history_wf, test.iloc[[i]]])
        except Exception:
            continue

    sarimax_preds_log = np.array(sarimax_preds_log)
    X_feat     = np.array(feat_rows)
    actual_arr = np.array(actual_vals)

    pred_xgb_nt = np.exp(sarimax_preds_log + xgb_h_nt.predict(X_feat))
    pred_lgb_nt = np.exp(sarimax_preds_log + lgb_h_nt.predict(X_feat))

    res_xgb_nt = evaluate_forecast(actual_arr, pred_xgb_nt)
    res_lgb_nt = evaluate_forecast(actual_arr, pred_lgb_nt)

    all_results_notune.append({
        'horizon': h, 'actual': actual_arr,
        'xgb_preds': pred_xgb_nt, 'lgb_preds': pred_lgb_nt,
        'hybrid_xgb': res_xgb_nt, 'hybrid_lgb': res_lgb_nt,
    })
    metrics_rows_notune.extend([
        {'horizon': f'H+{h}', 'model': 'Hybrid XGB (no-tune)', 'MAE': res_xgb_nt['mae'], 'RMSE': res_xgb_nt['rmse'], 'R2': res_xgb_nt['r2']},
        {'horizon': f'H+{h}', 'model': 'Hybrid LGB (no-tune)', 'MAE': res_lgb_nt['mae'], 'RMSE': res_lgb_nt['rmse'], 'R2': res_lgb_nt['r2']},
    ])
    print(f"H+{h} | XGB(nt) MAE:{res_xgb_nt['mae']:>10,.0f} R2:{res_xgb_nt['r2']:>6.3f} | LGB(nt) MAE:{res_lgb_nt['mae']:>10,.0f} R2:{res_lgb_nt['r2']:>6.3f}")

(Tanpa Tuning) Horizon H+1
H+1 | XGB(nt) MAE:    12,024 R2: 0.997 | LGB(nt) MAE:     9,941 R2: 0.998
(Tanpa Tuning) Horizon H+2
H+2 | XGB(nt) MAE:    17,652 R2: 0.996 | LGB(nt) MAE:    20,384 R2: 0.995
(Tanpa Tuning) Horizon H+3
H+3 | XGB(nt) MAE:    26,482 R2: 0.993 | LGB(nt) MAE:    24,251 R2: 0.994
(Tanpa Tuning) Horizon H+4
H+4 | XGB(nt) MAE:    25,373 R2: 0.993 | LGB(nt) MAE:    28,271 R2: 0.992
(Tanpa Tuning) Horizon H+5
H+5 | XGB(nt) MAE:    30,513 R2: 0.991 | LGB(nt) MAE:    31,523 R2: 0.991
(Tanpa Tuning) Horizon H+6
H+6 | XGB(nt) MAE:    47,642 R2: 0.983 | LGB(nt) MAE:    37,317 R2: 0.988
(Tanpa Tuning) Horizon H+7
H+7 | XGB(nt) MAE:    42,432 R2: 0.985 | LGB(nt) MAE:    47,970 R2: 0.981


## Export Metrics


In [ ]:
# export metrics Experiment 1
metrics_exp1 = pd.DataFrame(metrics_rows_notune)
metrics_exp1 = metrics_exp1.sort_values(['horizon', 'model']).reset_index(drop=True)
metrics_exp1.to_csv('metrics_exp1_no_tune.csv', index=False)
metrics_exp1


,horizon,model,MAE,RMSE,R2
0,H+1,Hybrid LGB (no-tune),9940.516133,20240.207426,0.998097
1,H+1,Hybrid XGB (no-tune),12024.120357,23420.185037,0.997452
2,H+2,Hybrid LGB (no-tune),20383.732558,32389.991079,0.995121
3,H+2,Hybrid XGB (no-tune),17652.004221,29761.286589,0.995881
4,H+3,Hybrid LGB (no-tune),24250.543781,35960.031081,0.993980
5,H+3,Hybrid XGB (no-tune),26482.472726,39005.193541,0.992917
6,H+4,Hybrid LGB (no-tune),28271.127497,40542.843478,0.992339
7,H+4,Hybrid XGB (no-tune),25373.249559,37385.444777,0.993486
8,H+5,Hybrid LGB (no-tune),31523.478075,44581.664339,0.990727
9,H+5,Hybrid XGB (no-tune),30512.508514,42832.600681,0.991440
